# ETL de Universidades utilizando API

Este projeto demonstra um pequeno pipeline de dados utilizando o conceito de **ETL (Extract, Transform, Load)**.

Os dados são extraídos de uma API pública de universidades e armazenados em um **banco de dados relacional (SQLite)** para permitir consultas futuras, histórico e integração com outras aplicações.

API utilizada:
http://universities.hipolabs.com

In [3]:
import requests
import sqlite3
import pandas as pd

In [4]:
# Extraindo dados da API
url = "http://universities.hipolabs.com/search?country=Brazil"

response = requests.get(url)

data = response.json()

print(f"Quantidade de universidades retornadas: {len(data)}")

Quantidade de universidades retornadas: 190


In [5]:
# Transformando o JSON em DataFrame
df = pd.DataFrame(data)

# Selecionando colunas relevantes
df = df[["name", "country", "domains", "web_pages"]]

df.head()

,name,country,domains,web_pages
0,Universidade Comunitária da Região de Chapecó ...,Brazil,[unochapeco.edu.br],[https://unochapeco.edu.br]
1,"Centro Universitário de Brasília, UNICEUB",Brazil,"[sempreceub.com, uniceub.br]",[https://www.uniceub.br]
2,Centro Universitário Barao de Maua,Brazil,[baraodemaua.br],[http://www.baraodemaua.br/]
3,Universidade Braz Cubas,Brazil,[brazcubas.br],[http://www.brazcubas.br/]
4,Universidade Candido Mendes,Brazil,[candidomendes.br],[http://www.candidomendes.br/]


In [6]:
# Criando conexão com banco
conn = sqlite3.connect("universities.db")
cursor = conn.cursor()

## Modelagem do Banco de Dados

Para armazenar os dados retornados pela API foi criada a tabela `universities`.

Estrutura da tabela:

| Coluna | Tipo | Descrição |
|------|------|------|
| id | INTEGER | Identificador da universidade |
| name | TEXT | Nome da universidade |
| country | TEXT | País da universidade |
| domain | TEXT | Domínio da universidade |
| web_page | TEXT | Site da universidade |

Essa modelagem foi escolhida por ser simples e suficiente para armazenar os dados retornados pela API, permitindo consultas futuras e integração com outras aplicações.

In [7]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS universities (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    country TEXT,
    domain TEXT,
    web_page TEXT
)
""")

conn.commit()

In [8]:
# Inserindo dados no banco

for _, row in df.iterrows():
    
    domain = row["domains"][0] if row["domains"] else None
    webpage = row["web_pages"][0] if row["web_pages"] else None

    cursor.execute("""
    INSERT INTO universities (name, country, domain, web_page)
    VALUES (?, ?, ?, ?)
    """, (row["name"], row["country"], domain, webpage))

conn.commit()

In [9]:
query = "SELECT * FROM universities LIMIT 10"

pd.read_sql(query, conn)

,id,name,country,domain,web_page
0,1,Universidade Comunitária da Região de Chapecó ...,Brazil,unochapeco.edu.br,https://unochapeco.edu.br
1,2,"Centro Universitário de Brasília, UNICEUB",Brazil,sempreceub.com,https://www.uniceub.br
2,3,Centro Universitário Barao de Maua,Brazil,baraodemaua.br,http://www.baraodemaua.br/
3,4,Universidade Braz Cubas,Brazil,brazcubas.br,http://www.brazcubas.br/
4,5,Universidade Candido Mendes,Brazil,candidomendes.br,http://www.candidomendes.br/
5,6,Universidade Castelo Branco,Brazil,castelobranco.br,http://www.castelobranco.br/
6,7,Centro Universitário Claretiano,Brazil,claretiano.edu.br,http://www.claretiano.edu.br/
7,8,Centro Regional Universitário de Espiríto Sant...,Brazil,creupi.br,http://www.creupi.br/
8,9,EMESCAM - Escola Superior de Ciências da Santa...,Brazil,emescam.br,http://www.emescam.br/
9,10,Universidade Federal de São Paulo,Brazil,epm.br,http://www.epm.br/


## Estratégia para armazenar dados de múltiplos países

Quando a API retorna dados de vários países, a melhor estratégia é manter a coluna `country` na tabela de universidades ou criar uma tabela separada de países relacionada por chave estrangeira.

Uma abordagem recomendada seria:

Tabela `countries`
- id
- country_name

Tabela `universities`
- id
- name
- domain
- web_page
- country_id

Essa estrutura evita duplicação de dados, melhora a organização do banco e facilita consultas por país.

Além disso, permite melhor manutenção do banco e escalabilidade caso novos dados sejam adicionados no futuro.